# 00 · Аудит вхідних даних та провенанс

**Питання цього нотебука: які саме файли дали результат корректора
EGG2015→EVRF2019?**

Друкує SHA-256 усіх вхідних растрів і рядів, версію та DOI ATL13, таблицю
вертикальних систем відліку з припливною конвенцією, лічильники рядків і
покриття, координати гідропостів, екстенти растрів, а також окремо **вхідні
дані по пониззю / Херсону**. Наприкінці записує
`outputs/reports/provenance.json`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
})
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

from kakhovka_altimetry.config import load_config
cfg = load_config()
TABLES = cfg.tables_dir
FIGS = cfg.repo_root / "outputs" / "figures"; FIGS.mkdir(parents=True, exist_ok=True)

import hashlib, json, platform, subprocess, datetime as dt

def sha256(path, _bufsize=1 << 20):
    path = pathlib.Path(path)
    if not path.exists():
        return None
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(_bufsize), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dir(d, pattern="*.parquet"):
    d = pathlib.Path(d)
    if not d.exists():
        return None
    h = hashlib.sha256()
    for p in sorted(d.rglob(pattern)):
        h.update(p.name.encode()); h.update(bytes.fromhex(sha256(p)))
    return h.hexdigest()

## 1 · Хеші вхідних файлів

In [2]:
gauge_dir = cfg.gauges.raw_dir
kherson = cfg.gauges.downstream_station(80805)
mykolaiv = cfg.gauges.downstream_station(98027)
inputs = {
    "egg2015_grid":        cfg.egg2015.path,
    "epsg9902_grid":       cfg.official_transform.path,
    "atl13_evrs_parquet":  cfg.evrs_parquet,
    "atl13_pass_levels":   cfg.pass_levels_parquet,
    "kherson_evrs_parquet":        cfg.downstream_body("kherson").evrs_parquet,
    "kherson_yearbook_csv":        kherson.yearbook_csv,
    "dnipro_estuary_evrs_parquet": cfg.downstream_body("dnipro_estuary").evrs_parquet,
    "dnipro_estuary_polygon":      cfg.downstream_body("dnipro_estuary").clip_polygon,
    "mykolaiv_2021_grid_csv":      mykolaiv.yearbook_grid_csv,
}
rows = []
for name, path in inputs.items():
    p = pathlib.Path(path)
    rows.append({"input": name, "path": str(p.relative_to(cfg.repo_root)) if p.exists() else str(path),
                 "exists": p.exists(),
                 "bytes": p.stat().st_size if p.exists() else None,
                 "sha256": sha256(p)})
rows.append({"input": "gauge_parquet_tree", "path": str(gauge_dir), "exists": gauge_dir.exists(),
             "bytes": None, "sha256": sha256_dir(gauge_dir)})
hash_tbl = pd.DataFrame(rows)
hash_tbl

,input,path,exists,bytes,sha256
0,egg2015_grid,data/1_data/egg_2015.tif,True,207381966.0,933cb8372fa5832b76d2acefba115c34c07af225bc9ec1...
1,epsg9902_grid,data/external/datum/ua_2019z.asc,True,114406.0,297440243e2ed8d67b1a61c6099db173b5fb13f8597385...
2,atl13_evrs_parquet,data/processed/kakhovka_atl13_evrs.parquet,True,31750730.0,b919f20970501283854f9f9e76ace2a79cfa3def34ab86...
3,atl13_pass_levels,data/processed/kakhovka_atl13_pass_levels.parquet,True,241173.0,60031306c1d9d6d9d4fb5c812ad26dbf22373565c12937...
4,kherson_evrs_parquet,data/processed/kherson_atl13_evrs.parquet,True,8066613.0,5e469ee359874db77047db3db50ec803cf86225480f930...
5,kherson_yearbook_csv,data/1_data/data/csv/dm_H/80805_yearbook.csv,True,129170.0,8dedbb1599e038c0fbd2c81702069819b9070a2eb105f4...
6,dnipro_estuary_evrs_parquet,data/processed/dnipro_estuary_atl13_evrs.parquet,True,426414991.0,c724517d89ed22d7935a6f41450c63af1d2766fcbf242b...
7,dnipro_estuary_polygon,data/aoi/dnipro_estuary.geojson,True,4221.0,74e0665371075e1375eda8fc85d4ef91c46ac4a2f5dec5...
8,mykolaiv_2021_grid_csv,config/mykolaiv_98027_2021_yearbook_grid.csv,True,2282.0,49b4a0624f9700907d32cdf9c711411e7083cf7e8fd44d...
9,gauge_parquet_tree,/home/niko/projects/icesat2-atl13-kakhovka/dat...,True,NaN,b03e8f27459f9ad38c6cba6227427ee7c4a1e2719fef67...


## 2 · Провенанс ATL13

In [3]:
from kakhovka_altimetry import discovery
manifest = discovery.read_manifest(cfg)
evrs = pd.read_parquet(cfg.evrs_parquet, columns=["time", "rgt", "period"])
evrs["time"] = pd.to_datetime(evrs["time"], utc=True)

atl13_prov = {
    "product": cfg.product.short_name,
    "version": cfg.product.version,
    "doi": "10.5067/ATLAS/ATL13.007",
    "n_granules_manifest": int(manifest["granule"].nunique()),
    "n_rgts": int(evrs["rgt"].nunique()),
    "date_min": str(evrs["time"].min().date()),
    "date_max": str(evrs["time"].max().date()),
    "n_segments": int(len(evrs)),
    "periods": evrs["period"].value_counts().to_dict(),
}
for body in cfg.downstream_bodies:
    m = body.manifest
    atl13_prov[f"n_granules_{body.slug}"] = (
        int(pd.read_csv(m)["granule"].nunique()) if m.exists() else 0
    )
atl13_prov

{'product': 'ATL13',
 'version': '007',
 'doi': '10.5067/ATLAS/ATL13.007',
 'n_granules_manifest': 323,
 'n_rgts': 17,
 'date_min': '2018-10-16',
 'date_max': '2025-12-31',
 'n_segments': 513873,
 'periods': {'PRE_BREACH': 401376,
  'POST_BREACH': 80001,
  'BREACH_DRAWDOWN': 32496},
 'n_granules_kherson': 177,
 'n_granules_dnipro_estuary': 266}

## 3 · Вертикальні системи відліку та припливна конвенція

EVRF2019 зрівняно в **zero-tide** системі (BKG). EGG2015 в цьому проєкті
трактується як EVRF2007-сумісний гравіметричний квазігеоїд (`config/vertical_datums.yaml`);
рамкові члени `evrf2019_minus_evrf2007_m` та `tide_system_correction_m` за
замовчуванням `0.0` означають **«не припущено»**, а не «нуль». Припливну
конвенцію EGG2015 **не** виводимо з імені файлу.

In [4]:
ot = cfg.official_transform
vref = pd.DataFrame([
    {"quantity": "ATL13 ht_water_surf", "frame": "WGS84 ellipsoid", "tide_system": "record from ICESat-2 docs (not assumed here)"},
    {"quantity": "EGG2015 zeta",        "frame": cfg.egg2015.height_system, "tide_system": "verify from source documentation"},
    {"quantity": "EPSG:%d grid" % ot.epsg_operation, "frame": ot.target_frame, "tide_system": ot.tide_system},
    {"quantity": "EVRF2019 (target)",   "frame": "EVRF2019", "tide_system": "zero_tide (BKG)"},
])
frame_terms = {
    "evrf2019_minus_evrf2007_m": ot.evrf2019_minus_evrf2007_m,
    "tide_system_correction_m": ot.tide_system_correction_m,
    "note": "0.0 == 'not assumed', kept as visible budget lines",
}
display(vref); frame_terms

,quantity,frame,tide_system
0,ATL13 ht_water_surf,WGS84 ellipsoid,record from ICESat-2 docs (not assumed here)
1,EGG2015 zeta,EVRF2007,verify from source documentation
2,EPSG:9902 grid,EVRF2019,zero-tide
3,EVRF2019 (target),EVRF2019,zero_tide (BKG)


{'evrf2019_minus_evrf2007_m': 0.0,
 'tide_system_correction_m': 0.0,
 'note': "0.0 == 'not assumed', kept as visible budget lines"}

## 4 · Покриття, координати, екстенти растрів

In [5]:
from kakhovka_altimetry import official_datum as od
from kakhovka_altimetry.gauges import load_gauge_observations

obs = load_gauge_observations(cfg)
gauge_cov = (obs.groupby("station_id")["datetime"]
             .agg(["min", "max", "size"])
             .rename(columns={"min": "start", "max": "end", "size": "n_obs"})
             .reset_index())
gauge_cov["slug"] = gauge_cov["station_id"].map(
    {s.id: s.slug for s in cfg.gauges.stations})
station_xy = pd.DataFrame([
    {"slug": s.slug, "name_en": s.name_en, "id": s.id, "lat": s.lat, "lon": s.lon,
     "gauge_zero_bs77_m": s.gauge_zero_baltic_m, "active_regimes": ",".join(s.active_regimes)}
    for s in cfg.gauges.stations])

grid = od.load_official_grid(cfg)
raster_ext = pd.DataFrame([
    {"raster": "EPSG:%d ua_2019z.asc" % ot.epsg_operation,
     "lon_min": float(grid.lons.min()), "lon_max": float(grid.lons.max()),
     "lat_min": float(grid.lats.min()), "lat_max": float(grid.lats.max()),
     "shape": str(grid.values.shape)},
])
display(station_xy); display(gauge_cov); raster_ext

,slug,name_en,id,lat,lon,gauge_zero_bs77_m,active_regimes
0,rozumivka,Rozumivka,80959,47.771208,35.148890,12.0,"PRE_BREACH,BREACH_DRAWDOWN,POST_BREACH"
1,plavni,Plavni,80961,47.567110,35.326056,12.0,PRE_BREACH
2,blahovishchenka,Blahovishchenka,80963,47.463877,34.821044,12.0,PRE_BREACH
3,nikopol,Nikopol,80964,47.554451,34.375355,12.0,PRE_BREACH
4,velyka_lepetykha,Velyka Lepetykha,80971,47.175331,33.931094,12.0,PRE_BREACH
5,nova_kakhovka,Nova Kakhovka,80977,46.775432,33.374615,12.0,PRE_BREACH


,station_id,start,end,n_obs,slug
0,80959,2020-01-01 06:00:00+00:00,2025-12-31 18:00:00+00:00,4346,rozumivka
1,80961,2020-01-01 06:00:00+00:00,2021-12-31 18:00:00+00:00,1462,plavni
2,80963,2020-01-01 06:00:00+00:00,2021-12-31 18:00:00+00:00,1462,blahovishchenka
3,80964,2020-01-01 06:00:00+00:00,2021-12-31 18:00:00+00:00,1462,nikopol
4,80971,2020-01-01 06:00:00+00:00,2021-12-31 18:00:00+00:00,1462,velyka_lepetykha
5,80977,2020-01-01 06:00:00+00:00,2021-12-31 18:00:00+00:00,1462,nova_kakhovka


,raster,lon_min,lon_max,lat_min,lat_max,shape
0,EPSG:9902 ua_2019z.asc,20.200559,45.100559,40.888296,55.588296,"(99, 167)"


## 5 · Вхідні дані по пониззю / Херсону

Херсон (пост 80805) — річковий пост Дніпра **нижче** греблі, без знівельованого
нуля. Ряд подається як стедж/локальна аномалія, **ніколи** як абсолютна WSE.

In [6]:
from kakhovka_altimetry.downstream import load_downstream_series, coverage_summary
ds = load_downstream_series(cfg)
display(coverage_summary(ds))

def _body_evrs_summary(body_slug: str) -> None:
    body = cfg.downstream_body(body_slug)
    kev = body.evrs_parquet
    if not kev.exists():
        print(f"no {body_slug} ATL13 pull yet -- run "
              f"scripts/download_atl13_downstream.py --body {body_slug}")
        return
    ke = pd.read_parquet(kev, columns=["time", "rgt", "period", "lon", "lat"])
    ke["time"] = pd.to_datetime(ke["time"], utc=True)
    print(f"{body_slug} ATL13 evrs:", len(ke), "segments,",
          ke["rgt"].nunique(), "RGTs,",
          f"{ke['time'].min().date()}..{ke['time'].max().date()}")
    print("  bbox lon %.3f..%.3f  lat %.3f..%.3f" % (
        ke.lon.min(), ke.lon.max(), ke.lat.min(), ke.lat.max()))
    kp = pd.read_parquet(body.pass_levels_parquet)
    display(kp.groupby("period")["n_points"].agg(
        passes="size", segments="sum").astype(int))

for _slug in ("kherson", "dnipro_estuary"):
    _body_evrs_summary(_slug)

,station_id,slug,name_en,start,end,n_days,years,sources,has_bs77,h_cm_min,h_cm_max,postbreach_h_cm_max
0,80805,kherson,Kherson,2019-01-01,2025-12-31,2176,"2019,2020,2021,2022,2023,2025","donor_parquet,yearbook_csv",True,415.0,1056.0,1056.0
1,98027,mykolaiv,Mykolaiv (Southern Bug),2019-01-01,2023-12-31,1461,"2019,2020,2021,2023","donor_parquet_versioned,yearbook_grid",True,407.0,596.0,596.0


kherson ATL13 evrs: 110649 segments, 11 RGTs, 2018-10-17..2026-01-02
  bbox lon 32.351..33.440  lat 46.513..47.899


,passes,segments
period,,
BREACH_DRAWDOWN,68,9416
POST_BREACH,245,22468
PRE_BREACH,544,61141


dnipro_estuary ATL13 evrs: 7048846 segments, 10 RGTs, 2018-10-17..2026-05-12


  bbox lon 30.671..33.190  lat 41.065..47.262


,passes,segments
period,,
BREACH_DRAWDOWN,81,27351
POST_BREACH,279,93231
PRE_BREACH,522,139211


## 6 · Вертикальне правило (не порушувати)

```
H_evrs_egg2015_m = ht_water_surf - zeta_EGG2015          (єдина фізична висота)
H_egm2008_m      = ht_ortho                              (лише контроль-стовпець)
```

`period` розколюється на 2023-06-06 (прорив греблі): PRE_BREACH / BREACH_DRAWDOWN /
POST_BREACH. Статистики водосховища ніколи не змішують PRE_BREACH з пост-проривом.

## 7 · Запис `provenance.json`

In [7]:
def git_commit():
    try:
        return subprocess.check_output(["git", "-C", str(cfg.repo_root), "rev-parse", "HEAD"],
                                       text=True).strip()
    except Exception:
        return None

prov = {
    "run_timestamp_utc": dt.datetime.now(dt.UTC).isoformat(),
    "git_commit": git_commit(),
    "python_version": platform.python_version(),
    "random_seed": 20260904,
    "atl13": atl13_prov,
    "sha256": {r["input"]: r["sha256"] for _, r in hash_tbl.iterrows()},
    "vertical_frame_terms_m": frame_terms,
    "pre_breach_cutoff": str(cfg.regimes.breach_start),
    # EPSG's operation-accuracy field for the 9902 transformation (NOT a sigma),
    # and, separately, the quoted EGG2015 model error. These are different terms;
    # they were previously both written under the epsg9902 key.
    "epsg9902_op_accuracy_m": cfg.official_transform.published_stats.get("accuracy_m"),
    "epsg9902_sd_at_determination_points_m": cfg.official_transform.published_stats.get("sd_m"),
    "egg2015_model_accuracy_m": cfg.official_transform.egg2015_accuracy_m,
    "bootstrap_resamples": 10000,
    "gauge_zero_bs77_m_nominal": 12.00,
    "downstream": {
        "bodies": {
            b.slug: {
                "refid": b.refid,
                "clip": (list(b.clip_bbox) if b.clip_bbox is not None
                         else str(b.clip_polygon.relative_to(cfg.repo_root))),
                "n_granules": int(pd.read_csv(b.manifest)["granule"].nunique())
                if b.manifest.exists() else 0,
            }
            for b in cfg.downstream_bodies
        },
        "gauge_series_days": {
            slug: int((ds["slug"] == slug).sum()) for slug in ("kherson", "mykolaiv")
        },
    },
}
out = cfg.reports_dir / "provenance.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(prov, indent=2, ensure_ascii=False), encoding="utf-8")
print("wrote", out.relative_to(cfg.repo_root))
print(json.dumps(prov, indent=1, ensure_ascii=False)[:1500])

wrote outputs/reports/provenance.json
{
 "run_timestamp_utc": "2026-09-04T15:40:29.034658+00:00",
 "git_commit": "b1c872044261b159bdb183e54279b87dd842c186",
 "python_version": "3.12.3",
 "random_seed": 20260904,
 "atl13": {
  "product": "ATL13",
  "version": "007",
  "doi": "10.5067/ATLAS/ATL13.007",
  "n_granules_manifest": 323,
  "n_rgts": 17,
  "date_min": "2018-10-16",
  "date_max": "2025-12-31",
  "n_segments": 513873,
  "periods": {
   "PRE_BREACH": 401376,
   "POST_BREACH": 80001,
   "BREACH_DRAWDOWN": 32496
  },
  "n_granules_kherson": 177,
  "n_granules_dnipro_estuary": 266
 },
 "sha256": {
  "egg2015_grid": "933cb8372fa5832b76d2acefba115c34c07af225bc9ec1d12a71d9b18e756d40",
  "epsg9902_grid": "297440243e2ed8d67b1a61c6099db173b5fb13f85973854198b167c6c67b1a95",
  "atl13_evrs_parquet": "b919f20970501283854f9f9e76ace2a79cfa3def34ab86b5c436f135564a10f5",
  "atl13_pass_levels": "60031306c1d9d6d9d4fb5c812ad26dbf22373565c12937cb1e006bc60d3ac257",
  "kherson_evrs_parquet": "5e469ee359